In [1]:
# Notebook 02b: Data Cleaning — Grocery Store Sales Dataset
# Author: Eric Lindolfo, B.Eng
# Source: Kaggle — Grocery Store Sales 2025 by Pratyush Puri
# Goal: Clean and prepare sales dataset for demand analysis and forecasting

import pandas as pd
import numpy as np

# Load raw sales dataset
sales = pd.read_csv('data/grocery_chain_data.csv')

print("Dataset loaded successfully!")
print("Shape:", sales.shape)

Dataset loaded successfully!
Shape: (1980, 11)


In [2]:
# Step 1: Explore column names and data types
print("Columns:", sales.columns.tolist())
print("\nFirst 5 rows:")
sales.head()

Columns: ['customer_id', 'store_name', 'transaction_date', 'aisle', 'product_name', 'quantity', 'unit_price', 'total_amount', 'discount_amount', 'final_amount', 'loyalty_points']

First 5 rows:


,customer_id,store_name,transaction_date,aisle,product_name,quantity,unit_price,total_amount,discount_amount,final_amount,loyalty_points
0,2824,GreenGrocer Plaza,2023-08-26,Produce,Pasta,2.0,7.46,14.92,0.00,14.92,377
1,5506,ValuePlus Market,2024-02-13,Dairy,Cheese,1.0,1.85,1.85,3.41,-1.56,111
2,4657,ValuePlus Market,2023-11-23,Bakery,Onions,4.0,7.38,29.52,4.04,25.48,301
3,2679,SuperSave Central,2025-01-13,Snacks & Candy,Cereal,3.0,5.50,16.50,1.37,15.13,490
4,9935,GreenGrocer Plaza,2023-10-13,Canned Goods,Orange Juice,5.0,8.66,43.30,1.50,41.80,22


In [3]:
# Step 2: Check data types and missing values
sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1980 entries, 0 to 1979
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customer_id       1980 non-null   int64  
 1   store_name        1955 non-null   object 
 2   transaction_date  1980 non-null   object 
 3   aisle             1980 non-null   object 
 4   product_name      1980 non-null   object 
 5   quantity          1980 non-null   float64
 6   unit_price        1980 non-null   float64
 7   total_amount      1980 non-null   float64
 8   discount_amount   1980 non-null   float64
 9   final_amount      1980 non-null   float64
 10  loyalty_points    1980 non-null   int64  
dtypes: float64(5), int64(2), object(4)
memory usage: 170.3+ KB


In [4]:
# Step 3: Statistical summary of numerical columns
sales.describe()

,customer_id,quantity,unit_price,total_amount,discount_amount,final_amount,loyalty_points
count,1980.000000,1980.000000,1980.000000,1980.000000,1980.000000,1980.000000,1980.000000
mean,5542.958081,2.968182,15.488045,45.902576,4.469591,41.432985,255.147980
std,2575.771856,1.419028,8.400823,35.018599,4.962001,32.593328,146.009333
min,1006.000000,1.000000,0.990000,1.010000,0.000000,-3.430000,0.000000
25%,3271.500000,2.000000,8.240000,18.000000,1.240000,15.800000,128.000000
50%,5582.500000,3.000000,15.190000,37.130000,3.045000,32.820000,265.500000
75%,7791.750000,4.000000,22.862500,67.930000,5.402500,60.800000,378.000000
max,9998.000000,5.000000,29.980000,149.900000,29.940000,147.910000,500.000000


In [5]:
# Step 4: Convert transaction_date to datetime
sales['transaction_date'] = pd.to_datetime(sales['transaction_date'])

# Step 5: Convert quantity from float to int
sales['quantity'] = sales['quantity'].astype(int)

# Step 6: Investigate missing store_name values
print("Missing store_name count:", sales['store_name'].isnull().sum())
print("\nRows with missing store_name:")
print(sales[sales['store_name'].isnull()][['customer_id', 'store_name', 'product_name', 'aisle']].head(10))

Missing store_name count: 25

Rows with missing store_name:
      customer_id store_name product_name              aisle
285          9445        NaN        Bread       Canned Goods
385          2983        NaN         Milk       Frozen Foods
389          9289        NaN       Cheese     Snacks & Candy
418          7658        NaN         Milk              Dairy
540          1472        NaN         Milk    Household Items
604          1919        NaN       Yogurt             Bakery
657          5835        NaN       Cereal  Health & Wellness
740          6327        NaN        Bread       Frozen Foods
868          8581        NaN      Bananas             Bakery
1111         3762        NaN       Yogurt             Bakery


In [6]:
# Step 7: Fill missing store_name with 'Unknown'
sales['store_name'] = sales['store_name'].fillna('Unknown')

print("Missing store_name after fix:", sales['store_name'].isnull().sum())

Missing store_name after fix: 0


In [7]:
# Step 8: Investigate negative final_amount values
# These occur when discount_amount > total_amount

negative = sales[sales['final_amount'] < 0]
print(f"Rows with negative final_amount: {len(negative)}")
print("\nSample:")
print(negative[['product_name', 'quantity', 'unit_price', 'total_amount', 'discount_amount', 'final_amount']].head())

# Step 9: Replace negative final_amount with 0
# Business logic: minimum transaction value is 0 — customer pays nothing
sales['final_amount'] = sales['final_amount'].clip(lower=0)

print(f"\nNegative values after fix: {(sales['final_amount'] < 0).sum()}")

Rows with negative final_amount: 13

Sample:
    product_name  quantity  unit_price  total_amount  discount_amount  \
1         Cheese         1        1.85          1.85             3.41   
28        Onions         1        2.46          2.46             4.71   
60        Yogurt         2        1.70          3.40             4.40   
665       Cereal         1        3.78          3.78             4.34   
773       Onions         1        1.01          1.01             4.44   

     final_amount  
1           -1.56  
28          -2.25  
60          -1.00  
665         -0.56  
773         -3.43  

Negative values after fix: 0


In [8]:
# Step 10: Export cleaned sales dataset
import os

os.makedirs('data/processed', exist_ok=True)

sales.to_csv('data/processed/sales_clean.csv', index=False)

print("Clean sales dataset exported successfully!")
print("Location: data/processed/sales_clean.csv")
print("\nFinal shape:", sales.shape)
print("\nData types:")
print(sales.dtypes)

Clean sales dataset exported successfully!
Location: data/processed/sales_clean.csv

Final shape: (1980, 11)

Data types:
customer_id                  int64
store_name                  object
transaction_date    datetime64[ns]
aisle                       object
product_name                object
quantity                     int64
unit_price                 float64
total_amount               float64
discount_amount            float64
final_amount               float64
loyalty_points               int64
dtype: object
